In [ ]:
import ast
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

from xgboost import XGBRegressor
import sys
from pathlib import Path

repo_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src" / "training" / "data_utils.py").is_file()
)
sys.path.insert(0, str(repo_root / "src"))
from training.data_utils import grouped_track_id_split


In [ ]:
artist_df = pd.read_csv("../../data/SpotGenTrack/Data Sources/spotify_artists.csv", index_col=0)
songs_df = pd.read_csv("../../data/SpotGenTrack/Data Sources/spotify_tracks.csv", index_col=0)

artist_df["genres"] = artist_df["genres"].apply(ast.literal_eval)

In [ ]:
genre_counts = Counter()

for genres in artist_df["genres"]:
    genre_counts.update(genres)

print("Unique genres:", len(genre_counts))
print(genre_counts.most_common(20))

In [ ]:
top_genres = [g for g, _ in genre_counts.most_common(100)]

for genre in top_genres:
    artist_df[f"genre_{genre}"] = artist_df["genres"].apply(
        lambda x: int(genre in x)
    )

In [ ]:
artist_features = (
    artist_df.groupby("track_id")
    .agg(
        artist_popularity=("artist_popularity", "mean"),
        followers=("followers", "mean"),
        num_artists=("id", "count"),
        **{
            f"genre_{genre}": (f"genre_{genre}", "max")
            for genre in top_genres
        }
    )
    .reset_index()
)

In [ ]:
df = pd.merge(
    songs_df,
    artist_features,
    left_on="id",
    right_on="track_id",
    how="inner"
)

In [ ]:
drop_cols = [
    "track_id",        # track ID from artist features
    "name",            # song name
    "track_name_prev", # duplicate song name
    "artists_id",      # artist ID
    "album_id",
    "analysis_url",
    "href",
    "preview_url",
    "track_href",
    "uri",
    "lyrics",
    "available_markets",
    "country",
    "type",
    "track_number",
    "disc_number",
    "playlist"
]

df = df.drop(columns=drop_cols, errors="ignore")

In [ ]:
print(df.select_dtypes(exclude=[np.number]).columns)

In [ ]:
target = "popularity"

track_ids = df["id"].astype(str)
X = df.drop(columns=[target, "id", "track_id"])
y = df[target]

train_ids, validation_ids, test_ids = map(
    set, grouped_track_id_split(track_ids.tolist())
)
train_mask = track_ids.isin(train_ids)
validation_mask = track_ids.isin(validation_ids)
test_mask = track_ids.isin(test_ids)

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_val, y_val = X.loc[validation_mask], y.loc[validation_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

print(X.shape)
print(y.shape)


In [ ]:
xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)

In [ ]:
# Predictions
y_train_pred = xgb.predict(X_train)
y_val_pred = xgb.predict(X_val)
y_test_pred = xgb.predict(X_test)

# R²
train_r2 = r2_score(y_train, y_train_pred)
val_r2 = r2_score(y_val, y_val_pred)
test_r2 = r2_score(y_test, y_test_pred)

# RMSE
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

# MAE
train_mae = mean_absolute_error(y_train, y_train_pred)
val_mae = mean_absolute_error(y_val, y_val_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

print(f"Train R2: {train_r2:.4f}")
print(f"Validation R2: {val_r2:.4f}")
print(f"Test R2: {test_r2:.4f}")

print(f"Train RMSE: {train_rmse:.4f}")
print(f"Validation RMSE: {val_rmse:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")

print(f"Train MAE: {train_mae:.4f}")
print(f"Validation MAE: {val_mae:.4f}")
print(f"Test MAE: {test_mae:.4f}")